# Load Dataset and Libraries


In [1]:
!pip install roboflow[]
%pip install ultralytics supervision roboflow
import ultralytics
ultralytics.checks()

Ultralytics 8.3.235 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (Tesla P100-PCIE-16GB, 16269MiB)
Setup complete ✅ (4 CPUs, 31.4 GB RAM, 6567.3/8062.4 GB disk)


provides tools for training, validating, and deploying YOLO 

In [3]:
import os

print("data.yaml exists:", os.path.exists("/kaggle/working/Smart-city(latest)-4/data.yaml"))

data.yaml exists: True


In [4]:
print("train/images exists:", os.path.exists("/kaggle/working/Smart-city(latest)-4/train/images"))
print("valid/images exists:", os.path.exists("/kaggle/working/Smart-city(latest)-4/valid/images"))

train/images exists: True
valid/images exists: True


# Dataset Description


In [6]:
import os
from collections import defaultdict

# Dataset location
dataset_location = "/kaggle/working/Smart-city(latest)-4"

# Class names
class_names = ['DefectedManhole', 'FullBin', 'Garbage', 'GoodManhole', 'ManagedBin', 'pothole']

# Folders to check
folders = ['train', 'valid', 'test']

def count_images_and_annotations(base_path, folder):
    """Count images and annotations per class"""
    images_path = os.path.join(base_path, folder, 'images')
    labels_path = os.path.join(base_path, folder, 'labels')
    
    # Count total images
    total_images = 0
    if os.path.exists(images_path):
        total_images = len([f for f in os.listdir(images_path) 
                           if f.endswith(('.jpg', '.jpeg', '.png'))])
    
    # Count annotations per class
    class_counts = defaultdict(int)
    total_annotations = 0
    
    if os.path.exists(labels_path):
        for label_file in os.listdir(labels_path):
            if label_file.endswith('.txt'):
                label_path = os.path.join(labels_path, label_file)
                with open(label_path, 'r') as f:
                    for line in f:
                        parts = line.strip().split()
                        if parts:
                            class_id = int(parts[0])
                            if 0 <= class_id < len(class_names):
                                class_counts[class_id] += 1
                                total_annotations += 1
    
    total_label_files = len([f for f in os.listdir(labels_path) 
                            if f.endswith('.txt')]) if os.path.exists(labels_path) else 0
    
    return total_images, total_label_files, class_counts, total_annotations

# Print header
print("=" * 80)
print("DATASET ANALYSIS - Smart City Dataset")
print("=" * 80)
print(f"\nClasses: {', '.join(class_names)}\n")

# Analyze each folder
for folder in folders:
    print("=" * 80)
    print(f"{folder.upper()} SET")
    print("=" * 80)
    
    total_images, total_labels, class_counts, total_annotations = count_images_and_annotations(
        dataset_location, folder
    )
    
    print(f"\nTotal Images: {total_images}")
    print(f"Total Label Files: {total_labels}")
    print(f"Total Annotations (objects): {total_annotations}")
    
    print(f"\nAnnotations per Class:")
    print("-" * 50)
    for class_id, class_name in enumerate(class_names):
        count = class_counts.get(class_id, 0)
        percentage = (count / total_annotations * 100) if total_annotations > 0 else 0
        print(f"  {class_name:20s}: {count:5d} ({percentage:5.2f}%)")
    
    print()

# Overall summary
print("=" * 80)
print("OVERALL SUMMARY")
print("=" * 80)

total_all_images = 0
total_all_annotations = 0
overall_class_counts = defaultdict(int)

for folder in folders:
    total_images, _, class_counts, total_annotations = count_images_and_annotations(
        dataset_location, folder
    )
    total_all_images += total_images
    total_all_annotations += total_annotations
    
    for class_id, count in class_counts.items():
        overall_class_counts[class_id] += count

print(f"\nTotal Images (all sets): {total_all_images}")
print(f"Total Annotations (all sets): {total_all_annotations}")

print(f"\nTotal Annotations per Class (across all sets):")
print("-" * 50)
for class_id, class_name in enumerate(class_names):
    count = overall_class_counts.get(class_id, 0)
    percentage = (count / total_all_annotations * 100) if total_all_annotations > 0 else 0
    print(f"  {class_name:20s}: {count:5d} ({percentage:5.2f}%)")

print("\n" + "=" * 80)

DATASET ANALYSIS - Smart City Dataset

Classes: DefectedManhole, FullBin, Garbage, GoodManhole, ManagedBin, pothole

TRAIN SET

Total Images: 2710
Total Label Files: 2710
Total Annotations (objects): 4614

Annotations per Class:
--------------------------------------------------
  DefectedManhole     :   574 (12.44%)
  FullBin             :   743 (16.10%)
  Garbage             :   973 (21.09%)
  GoodManhole         :   703 (15.24%)
  ManagedBin          :   739 (16.02%)
  pothole             :   882 (19.12%)

VALID SET

Total Images: 586
Total Label Files: 586
Total Annotations (objects): 1041

Annotations per Class:
--------------------------------------------------
  DefectedManhole     :   138 (13.26%)
  FullBin             :   178 (17.10%)
  Garbage             :   199 (19.12%)
  GoodManhole         :   148 (14.22%)
  ManagedBin          :   156 (14.99%)
  pothole             :   222 (21.33%)

TEST SET

Total Images: 578
Total Label Files: 578
Total Annotations (objects): 1002

Ann